<a href="https://colab.research.google.com/github/janithcyapa/Wearable-Biomechanics/blob/main/Dataset_Manager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wearable Biomechanics Dataset Manager


# Developemnt

#### Setup Environment

In [ ]:
# @title
!pip install duckdb pyarrow pandas


In [31]:
# @title
import duckdb
import json
import os
import io
import pandas as pd
from datetime import datetime
from google.colab import files
from google.colab import drive
from IPython.display import display

In [32]:
# @title
drive.mount('/content/drive', force_remount=True)
BASE_DIR = '/content/drive/MyDrive/Datasets/Wearable-Biomechanics'

Mounted at /content/drive


## Subject Manager


In [18]:
# @title SubjectManager
class SubjectManager:
    def __init__(self):
        # This targets the folder/shortcut in the mounted Drive
        self.base_dir = BASE_DIR
        self.file_path = f'{self.base_dir}/subjects.json'

        # Safety check: Create the folder if it somehow doesn't exist
        os.makedirs(self.base_dir, exist_ok=True)

        # Safety check: if the file doesn't exist, create an empty one
        if not os.path.exists(self.file_path):
            with open(self.file_path, 'w') as f:
                json.dump({}, f)
            print(f"Created a new subjects.json file in {self.base_dir}.")
        else:
            print(f"Connected to existing subjects.json in {self.base_dir}.")

    # --- HELPER FUNCTIONS ---
    def _load_data(self):
        with open(self.file_path, 'r') as f:
            return json.load(f)

    def _save_data(self, data):
        with open(self.file_path, 'w') as f:
            json.dump(data, f, indent=4)

    # --- 1. CREATE ---
    def add_subject(self, subject_id, name, weight, height, max_lift, description):
        data = self._load_data()

        if subject_id in data:
            print(f"⚠️ Error: Subject ID '{subject_id}' already exists!")
            return

        data[subject_id] = {
            "name": name,
            "weight_kg": weight,
            "height_cm": height,
            "max_lift_kg": max_lift,
            "description": description
        }
        self._save_data(data)
        print(f"✅ Successfully added {name} ({subject_id}).")

    # --- 2. READ ---
    def list_subjects(self, subject_id=None):
        data = self._load_data()

        if subject_id:
            return data.get(subject_id, "⚠️ Subject not found.")

        print(f"\n--- Database contains {len(data)} subjects ---")
        # for sub_id, info in data.items():
        #     print(f"ID: {sub_id} | Name: {info['name']} | Weight: {info['weight_kg']}kg")
        return data

    # --- 3. UPDATE ---
    def update_subject(self, subject_id, **kwargs):
        data = self._load_data()

        if subject_id not in data:
            print(f"⚠️ Error: Subject ID '{subject_id}' not found.")
            return

        for key, value in kwargs.items():
            data[subject_id][key] = value

        self._save_data(data)
        print(f"✅ Successfully updated {subject_id}.")

    # --- 4. DELETE ---
    def delete_subject(self, subject_id):
        data = self._load_data()

        if subject_id in data:
            del data[subject_id]
            self._save_data(data)
            print(f"🗑️ Successfully deleted {subject_id}.")
        else:
            print(f"⚠️ Error: Subject ID '{subject_id}' not found.")

# Initialize the manager
db = SubjectManager()

Connected to existing subjects.json in /content/drive/MyDrive/Datasets/Wearable-Biomechanics.


In [36]:
# @title List All Subjects
raw_data = db.list_subjects()
df = pd.DataFrame.from_dict(raw_data, orient='index')
df.reset_index(inplace=True)
df.rename(columns={'index': 'Subject ID', 'name': 'Name', 'weight_kg': 'Weight (kg)',
                   'height_cm': 'Height (cm)', 'max_lift_kg': 'Max Lift (kg)',
                   'description': 'Description'}, inplace=True)
display(df)


--- Database contains 1 subjects ---


,Subject ID,Name,Weight (kg),Height (cm),Max Lift (kg),Description
0,sub_001,John Doe,78.5,180,120,Right-handed


In [ ]:
# @title Helper Codes
db.add_subject(
    subject_id="sub_001",
    name="John Doe",
    weight=78.5,
    height=180,
    max_lift=120,
    description="Right-handed"
)
db.update_subject(
    "sub_001",
    name="John Doe",
    weight=78.5,
    height=180,
    max_lift=120,
    description="Right-handed"
)
db.delete_subject("sub_001")

In [24]:
# @title
db.add_subject(
    subject_id="sub_001",
    name="John Doe",
    weight=78.5,
    height=180,
    max_lift=120,
    description="Right-handed"
)

✅ Successfully added John Doe (sub_001).


## Experiment Manager

In [47]:
# @title
class ExperimentManager:
    def __init__(self):
        self.base_dir = BASE_DIR
        self.index_dir = f'{self.base_dir}/db_index'
        self.raw_dir = f'{self.base_dir}/raw_data'
        self.file_path = f'{self.index_dir}/experiments.json'

        os.makedirs(self.index_dir, exist_ok=True)
        os.makedirs(self.raw_dir, exist_ok=True)

        if not os.path.exists(self.file_path):
            with open(self.file_path, 'w') as f:
                json.dump({}, f)
            print("Created a new experiments.json file.")
        else:
            print("Connected to experiments database.")

    def _load_data(self):
        with open(self.file_path, 'r') as f:
            return json.load(f)

    def _save_data(self, data):
        with open(self.file_path, 'w') as f:
            json.dump(data, f, indent=4)

    # --- 1. UPLOAD EXPERIMENT ---
    def interactive_upload(self, experiment_id, subject_id, date, time, activity_type, description):
        data = self._load_data()

        if experiment_id in data:
            print(f"⚠️ Error: Experiment ID '{experiment_id}' already exists.")
            return

        print(f"📂 Please select the raw CSV file for {experiment_id}...")
        uploaded = files.upload()

        if not uploaded:
            print("⚠️ Upload cancelled.")
            return

        original_filename = list(uploaded.keys())[0]

        try:
            raw_dataframe = pd.read_csv(io.BytesIO(uploaded[original_filename]))
        except Exception as e:
            print(f"⚠️ Error reading CSV: {e}")
            return

        # Format date for filename (removes hyphens, e.g., 2026-05-06 -> 20260506)
        clean_date = date.replace("-", "").replace("/", "")
        new_filename = f"{subject_id}_{experiment_id}_{clean_date}.csv"

        csv_rel_path = f"raw_data/{new_filename}"
        csv_full_path = f"{self.raw_dir}/{new_filename}"

        raw_dataframe.to_csv(csv_full_path, index=False)

        # Log all the new metadata
        data[experiment_id] = {
            "subject_id": subject_id,
            "date": date,
            "time": time,
            "activity_type": activity_type,
            "description": description,
            "raw_csv_path": csv_rel_path,
            "processed_files": {}
        }

        self._save_data(data)
        del uploaded[original_filename]
        print(f"✅ Success! Data uploaded, renamed to '{new_filename}', and logged.")

    # --- 2. LIST EXPERIMENTS ---
    def list_experiments(self):
        data = self._load_data()
        if not data:
            print("Database is empty.")
            return

        # Flatten the dictionary so it looks beautiful in Pandas
        table_rows = []
        for exp_id, info in data.items():

            # Format the processed files nicely
            proc_files = info.get('processed_files', {})
            if proc_files:
                # Creates a string like: "joint_estimation: file.parquet | mesh: file2.parquet"
                proc_str = " | ".join([f"{tag}: {os.path.basename(path)}" for tag, path in proc_files.items()])
            else:
                proc_str = "❌ None yet"

            table_rows.append({
                'Experiment ID': exp_id,
                'Subject': info.get('subject_id', 'N/A'),
                'Date': info.get('date', 'N/A'),
                'Activity': info.get('activity_type', 'N/A'),
                'Raw File': info.get('raw_csv_path', 'N/A'),
                'Processed Files': proc_str
            })

        df = pd.DataFrame(table_rows)

        # Display the interactive table
        print(f"\n📊 Database contains {len(df)} experiments:")
        display(df)

    # --- 3. DELETE EXPERIMENT ---
    def delete_experiment(self, experiment_id):
        data = self._load_data()
        if experiment_id not in data:
            print(f"⚠️ Error: '{experiment_id}' not found.")
            return

        exp_info = data[experiment_id]
        print(f"🗑️ Initiating deletion for {experiment_id}...")

        raw_path = f"{self.base_dir}/{exp_info['raw_csv_path']}"
        if os.path.exists(raw_path):
            os.remove(raw_path)
            print(f"  - Deleted raw file: {exp_info['raw_csv_path']}")

        for tag, processed_path in exp_info['processed_files'].items():
            full_proc_path = f"{self.base_dir}/{processed_path}"
            if os.path.exists(full_proc_path):
                os.remove(full_proc_path)
                print(f"  - Deleted processed [{tag}] file.")

        del data[experiment_id]
        self._save_data(data)
        print(f"✅ Experiment '{experiment_id}' completely removed.")

    # --- 4. LOG PROCESSED DATA ---
    def add_processed_data(self, experiment_id, tag, processed_df):
        """
        Saves a processed dataframe as a Parquet file and links it to the experiment.
        tag: e.g., 'kalman_filtered' or 'gnn_posture_output'
        """
        data = self._load_data()

        if experiment_id not in data:
            print(f"⚠️ Error: Experiment '{experiment_id}' not found.")
            return

        # 1. Create the specific subfolder for this tag (e.g., processed_data/kalman_filtered/)
        save_dir = f"{self.base_dir}/processed_data/{tag}"
        os.makedirs(save_dir, exist_ok=True)

        # 2. Define the exact file paths
        filename = f"{experiment_id}_{tag}.parquet"
        full_path = f"{save_dir}/{filename}"
        rel_path = f"processed_data/{tag}/{filename}"

        # 3. Save as Parquet (much faster and smaller than CSV for ML workflows)
        processed_df.to_parquet(full_path, engine='pyarrow', index=False)

        # 4. Update the JSON database
        data[experiment_id]['processed_files'][tag] = rel_path
        self._save_data(data)

        print(f"✅ Processed data '{tag}' saved and linked to {experiment_id}!")
# Initialize the manager
exp_db = ExperimentManager()

Connected to experiments database.


In [44]:
# @title
exp_db.interactive_upload(
    experiment_id="exp_001",
    subject_id="sub_001",
    date="2026-05-06",
    time="14:30",
    activity_type="fast_walking",
    description="IMU placed on lower back. Backpack weight: 5kg."
)

📂 Please select the raw CSV file for exp_001...


Saving combined_log_wide_1759324207.csv to combined_log_wide_1759324207 (2).csv
✅ Success! Data uploaded, renamed to 'sub_001_exp_001_20260506.csv', and logged.


In [51]:
exp_db.list_experiments()


📊 Database contains 1 experiments:


,Experiment ID,Subject,Date,Activity,Raw File,Processed Files
0,exp_001,sub_001,2026-05-06,fast_walking,raw_data/sub_001_exp_001_20260506.csv,kalman_estimates: exp_001_kalman_estimates.par...


In [42]:
# @title
exp_db.delete_experiment(
    experiment_id="exp_001"
)

🗑️ Initiating deletion for exp_001...
  - Deleted raw file: raw_data/sub_001_exp_001_20260506.csv
✅ Experiment 'exp_001' completely removed.


In [50]:
TARGET_EXP = "exp_001"
RAW_FILE_PATH = f"{exp_db.base_dir}/raw_data/sub_001_exp_001_20260506.csv"
df_raw = pd.read_csv(RAW_FILE_PATH)

df_filtered = df_raw.copy()
exp_db.add_processed_data(
    experiment_id=TARGET_EXP,
    tag="gnn_posture",
    processed_df=df_filtered
)

✅ Processed data 'gnn_posture' saved and linked to exp_001!
